In [1]:
!pip install vk_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 kB 2.4 MB/s eta 0:00:00


In [14]:
import vk_api
import datetime
import json
import csv
import time

TOKEN = ''
GROUP_DOMAIN = 'samara_university'

START_DATE = datetime.datetime(2026, 8, 1, 0, 0, 0)
END_DATE = datetime.datetime(2026, 8, 31, 23, 59, 59)

start_ts = int(START_DATE.timestamp())
end_ts = int(END_DATE.timestamp())

def main():
    vk_session = vk_api.VkApi(token=TOKEN)
    vk = vk_session.get_api()

    all_posts = []
    offset = 0
    count = 100
    total_fetched = 0
    stop_flag = False

    while not stop_flag:
        try:
            response = vk.wall.get(
                domain=GROUP_DOMAIN,
                count=count,
                offset=offset,
                extended=0,
                v=5.199
            )

            items = response['items']
            if not items:
                break

            for post in items:
                post_date_ts = post['date']

                if post_date_ts < start_ts:
                    stop_flag = True
                    break

                if start_ts <= post_date_ts <= end_ts:
                    owner_id = post['owner_id']          # отрицательный для сообществ
                    post_id = post['id']
                    post_url = f"https://vk.com/{GROUP_DOMAIN}?w=wall{owner_id}_{post_id}"

                    post_data = {
                        'id': post_id,
                        'owner_id': owner_id,
                        'url': post_url,
                        'date': datetime.datetime.fromtimestamp(post_date_ts).strftime('%Y-%m-%d %H:%M:%S'),
                        'text': post.get('text', ''),
                        'likes': post.get('likes', {}).get('count', 0),
                        'comments': post.get('comments', {}).get('count', 0),
                        'reposts': post.get('reposts', {}).get('count', 0),
                        'views': post.get('views', {}).get('count', 0)
                    }
                    all_posts.append(post_data)
                    total_fetched += 1

            print(f"Обработано постов: {offset + len(items)}. Собрано за август: {total_fetched}")

            if not stop_flag:
                offset += count
                time.sleep(0.34)

        except vk_api.exceptions.ApiError as e:
            print(f"Ошибка API: {e}")
            break
        except Exception as e:
            print(f"Неизвестная ошибка: {e}")
            break

    json_filename = 'samara_university_august_2026.json'
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(all_posts, f, ensure_ascii=False, indent=4)
    print(f"JSON сохранён: {json_filename} ({len(all_posts)} постов)")

    csv_filename = 'samara_university_august_2026.csv'
    fieldnames = ['id', 'owner_id', 'url', 'date', 'text', 'likes', 'comments', 'reposts', 'views']
    with open(csv_filename, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_posts)
    print(f"CSV сохранён: {csv_filename} ({len(all_posts)} постов)")

if __name__ == '__main__':
    main()

Обработано постов: 100. Собрано за август: 0
Обработано постов: 200. Собрано за август: 40
JSON сохранён: samara_university_august_2026.json (40 постов)
CSV сохранён: samara_university_august_2026.csv (40 постов)
